# 08 — Recovery Route Inventory (read-only diagnostic)

**Purpose.** Test three RNA-recovery routes that notebooks 06/07 did **not** evaluate because
those notebooks only matched exact `sample_id` values against on-disk expression files.

| Route | Mechanism | This notebook |
|---|---|---|
| **B** | Cell-line-name → DepMap `ModelID` join (`Model.csv`) | Executed |
| **A** | de Nonneville public deposit (ArrayExpress) | Documented only — no download |
| **C** | Lawlor internal PDX RNA | Documented only — requires lab access |

**Read-only.** Does not modify the manifest, does not train models, does not load full
expression matrices. Writes `data/processed/notebook_08_report.md`.

**Output:** `data/processed/notebook_08_report.md`

In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ \u2014 tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-18T16:43:12Z


## 1 — Load manifest and shared DepMap lookups

In [2]:
from src.manifest_builder import load_altitude_cellines
from src.aim1_inventory import DEPMAP_RNA_PATH, depmap_rna_model_ids
from src.recovery_inventory import (
    MODEL_CSV_PATH,
    build_depmap_name_lookup,
    select_route_b_candidates,
    build_route_b_table,
    route_b_alt_rna_counts,
)

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest)}')

model_df = pd.read_csv(MODEL_CSV_PATH, dtype=str)
lookup = build_depmap_name_lookup(model_df)
depmap_rna_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
print(f'Model.csv rows: {len(model_df)}')
print(f'DepMap RNA ModelIDs: {len(depmap_rna_models)}')

altitude = load_altitude_cellines()
altitude_ids = set(altitude['sample_id'].astype(str))
print(f'ALTitude panel rows: {len(altitude)}  unique sample_id: {len(altitude_ids)}')

Manifest rows: 115
Model.csv rows: 2154
DepMap RNA ModelIDs: 1719
ALTitude panel rows: 959  unique sample_id: 958


## 2 — Route B: cell-line-name → DepMap ModelID join

Candidates: every `Taglialatela_2026` manifest row **plus** any manifest row whose
`sample_id` appears in the ALTitude DepMap panel (31 osteosarcoma DepMap lines in the
current manifest, even when `alt_source` does not mention ALTitude).

Matching uses `Model.csv` columns `CellLineName`, `StrippedCellLineName`, `CCLEName`,
and `ModelIDAlias` with case/punctuation normalization; a prefix fallback on
`StrippedCellLineName` handles names like `G292` → `G292CLONEA141B1`.

RNA backing: mapped `ModelID` must appear in the DepMap expression matrix index
(`ModelID` column only — no full matrix load).

In [3]:
route_b_candidates = select_route_b_candidates(manifest, altitude_ids)
print(f'Route B candidates: {len(route_b_candidates)}')
print(route_b_candidates['route_b_reason'].value_counts().to_string())
print()

route_b = build_route_b_table(route_b_candidates, lookup, model_df, depmap_rna_models)
rb_counts = route_b_alt_rna_counts(route_b)

display_cols = [
    'cell_line_name', 'mapped_model_id', 'in_depmap_rna',
    'alt_status', 'source', 'cohort', 'match_method',
]
with pd.option_context('display.max_rows', None, 'display.width', 200):
    print(route_b[display_cols].to_string(index=False))

print()
print('=== Route B summary ===')
print(f"  Mapped to a ModelID:           {rb_counts['n_mapped_total']}")
print(f"  ModelID in DepMap RNA matrix:  {rb_counts['n_in_depmap_rna']}")
print(f"  ALT-positive + RNA-backed:     {rb_counts['n_alt_positive_rna']}")
print(f"  ALT-negative + RNA-backed:     {rb_counts['n_alt_negative_rna']}")
print(f"  Other alt_status + RNA-backed: {rb_counts['n_alt_other_rna']}")

unmapped = route_b[~route_b['mapped_model_id'].astype(bool)]
if len(unmapped):
    print(f"\n!! {len(unmapped)} candidate(s) failed to map to any ModelID:")
    for _, r in unmapped.iterrows():
        print(f"   - {r['cell_line_name']}  (manifest {r['manifest_sample_id']})")

mapped_not_rna = route_b[route_b['mapped_model_id'].astype(bool) & ~route_b['in_depmap_rna']]
if len(mapped_not_rna):
    print(f"\n!! {len(mapped_not_rna)} mapped ModelID(s) NOT in DepMap RNA file:")
    for _, r in mapped_not_rna.iterrows():
        print(f"   - {r['cell_line_name']} -> {r['mapped_model_id']}")

Route B candidates: 38
route_b_reason
ALTitude_DepMap_panel    31
Taglialatela_2026         7

 cell_line_name mapped_model_id  in_depmap_rna             alt_status                                                   source            cohort                 match_method
G292CLONEA141B1      ACH-000082           True            alt_unknown                                 pending_literature_merge       DepMap_26Q1     sample_id(exact ModelID)
           MG63      ACH-000359           True            alt_unknown                                 pending_literature_merge       DepMap_26Q1     sample_id(exact ModelID)
           U2OS      ACH-000364           True            alt_unknown                                 pending_literature_merge       DepMap_26Q1     sample_id(exact ModelID)
          SAOS2      ACH-000410           True            alt_unknown                                 pending_literature_merge       DepMap_26Q1     sample_id(exact ModelID)
            HOS      ACH-000613    

## 3 — Route A: de Nonneville expression availability (no download)

Public deposit: **ArrayExpress E-MTAB-11743**. A human must verify whether the
processed matrices are RNA-seq (usable for DE/classifier training) or methylation
array (not interchangeable with expression-based aims).

In [4]:
from src.recovery_inventory import summarize_de_nonneville, ARRAYEXPRESS_DE_NONNEVILLE

route_a = summarize_de_nonneville(manifest)
print('=== Route A: de Nonneville (labels only on disk) ===')
print(f"  Manifest samples:     {route_a['n_total']}")
print(f"  ALT-positive:         {route_a['n_alt_positive']}")
print(f"  ALT-negative:         {route_a['n_alt_negative']}")
print(f"  Other alt_status:     {route_a['n_other_alt_status']}")
print(f"  ArrayExpress accession: {route_a['arrayexpress_accession']}")

ROUTE_A_TODO = f"""
TODO — Route A (de Nonneville / {ARRAYEXPRESS_DE_NONNEVILLE})
[ ] Download {ARRAYEXPRESS_DE_NONNEVILLE} from ArrayExpress (do NOT assume assay type).
[ ] Open sample metadata: confirm modality is RNA-seq (not 450K methylation alone).
[ ] Map ArrayExpress sample names to these manifest sample_ids:
      ALT-positive ({route_a['n_alt_positive']}): {', '.join(route_a['sample_ids_positive'])}
      ALT-negative ({route_a['n_alt_negative']}): {', '.join(route_a['sample_ids_negative'])}
[ ] If RNA-seq confirmed: add expression file to data/raw/ and register in EXPRESSION_SOURCES.
[ ] Re-run notebooks 07 and 08 after integration.
"""
print(ROUTE_A_TODO)

=== Route A: de Nonneville (labels only on disk) ===
  Manifest samples:     22
  ALT-positive:         16
  ALT-negative:         6
  Other alt_status:     0
  ArrayExpress accession: E-MTAB-11743

TODO — Route A (de Nonneville / E-MTAB-11743)
[ ] Download E-MTAB-11743 from ArrayExpress (do NOT assume assay type).
[ ] Open sample metadata: confirm modality is RNA-seq (not 450K methylation alone).
[ ] Map ArrayExpress sample names to these manifest sample_ids:
      ALT-positive (16): deNonneville_1941.0, deNonneville_2254.0, deNonneville_2523.0, deNonneville_3103.0, deNonneville_4001.0, deNonneville_4078.0, deNonneville_4435.0, deNonneville_4462.0, deNonneville_4591.0, deNonneville_4665.0, deNonneville_4680.0, deNonneville_5136.0, deNonneville_7695.0, deNonneville_8299.0, deNonneville_8781.0, deNonneville_9496.0
      ALT-negative (6): deNonneville_5090.0, deNonneville_5664.0, deNonneville_6500.0, deNonneville_6537.0, deNonneville_6699.0, deNonneville_7855.0
[ ] If RNA-seq confirmed: 

## 4 — Route C: Lawlor internal PDX RNA (no public file)

Lawlor 2026 GCCRI PDX models carry peer-reviewed SV / IHC / ALT labels in the manifest
but ship **no** expression matrix in `data/raw/`.

In [5]:
from src.recovery_inventory import summarize_lawlor

route_c = summarize_lawlor(manifest)
print('=== Route C: Lawlor PDX (labels only on disk) ===')
print(f"  Total Lawlor manifest rows:          {route_c['n_total_lawlor']}")
print(f"  With ALT and/or architecture label:  {route_c['n_with_alt_or_architecture']}")
print(f"  ALT-positive:                        {route_c['n_alt_positive']}")
print(f"  ALT-negative:                        {route_c['n_alt_negative']}")
print(f"  tcab1 disrupted_high_conf:           {route_c['n_tcab1_disrupted_high_conf']}")
print(f"  Aim 1 pragmatic disrupted:           {route_c['n_aim1_pragmatic_disrupted']}")
print(f"  Aim 1 pragmatic intact:              {route_c['n_aim1_pragmatic_intact']}")

ROUTE_C_TODO = f"""
TODO — Route C (Lawlor internal PDX RNA — not public)
[ ] Request matched RNA-seq (raw or processed counts/TPM) from the Lawlor/GCCRI lab.
[ ] Samples needing matched RNA ({route_c['n_with_alt_or_architecture']} PDX models):
      All Lawlor sample_ids: {', '.join(route_c['sample_ids_all'])}
[ ] Priority for Aim 2 (ALT labels):
      ALT-positive ({route_c['n_alt_positive']}): {', '.join(route_c['sample_ids_alt_positive'])}
      ALT-negative ({route_c['n_alt_negative']}): {', '.join(route_c['sample_ids_alt_negative'])}
[ ] Priority for Aim 1 (architecture / TCAB1):
      Architecture-labeled: {', '.join(route_c['sample_ids_architecture_labeled'])}
[ ] On receipt: place files under data/raw/ (new subfolder) and register in EXPRESSION_SOURCES.
[ ] Re-run notebooks 06–08 after integration.
"""
print(ROUTE_C_TODO)

=== Route C: Lawlor PDX (labels only on disk) ===
  Total Lawlor manifest rows:          17
  With ALT and/or architecture label:  17
  ALT-positive:                        16
  ALT-negative:                        1
  tcab1 disrupted_high_conf:           4
  Aim 1 pragmatic disrupted:           4
  Aim 1 pragmatic intact:              0

TODO — Route C (Lawlor internal PDX RNA — not public)
[ ] Request matched RNA-seq (raw or processed counts/TPM) from the Lawlor/GCCRI lab.
[ ] Samples needing matched RNA (17 PDX models):
      All Lawlor sample_ids: 0522-000, 0560-000, 1922, 1981, OS33, OS37, OS39R, OS41, OS42, OS46, OS48, OS49, OS50, OS52, OS55, OS58, SJSA1
[ ] Priority for Aim 2 (ALT labels):
      ALT-positive (16): 0522-000, 0560-000, 1922, 1981, OS33, OS37, OS39R, OS41, OS42, OS46, OS48, OS49, OS50, OS52, OS55, OS58
      ALT-negative (1): SJSA1
[ ] Priority for Aim 1 (architecture / TCAB1):
      Architecture-labeled: 0560-000, OS33, OS37, OS39R
[ ] On receipt: place files unde

## 5 — Combined: Route-B-only best-case for Aim 1 and Aim 2

Assume Route B succeeds (DepMap join) and Routes A/C remain pending. File-backed RNA
means: direct `sample_id` in the DepMap RNA matrix **or** Route B maps the sample to
an in-matrix ModelID. All recovered expression uses DepMap log2(TPM+1) protein-coding.

Apply the same three-category verdict logic as notebooks 06/07.

In [6]:
from src.aim2_inventory import ALT_POSITIVE_LABEL, ALT_NEGATIVE_LABEL
from src.recovery_inventory import (
    build_route_b_only_aim1_inventory,
    build_route_b_only_aim2_inventory,
    assess_confounding_generic,
    decide_route_b_aim1,
    decide_route_b_aim2,
)
from src.aim1_inventory import WITHIN_COHORT_PER_ARM_MIN
from src.aim1_inventory import select_pragmatic_contrast_rows
from src.aim2_inventory import select_aim2_contrast_rows

aim1_rb = build_route_b_only_aim1_inventory(manifest, route_b)
aim2_rb = build_route_b_only_aim2_inventory(manifest, route_b)

print('=== Aim 1 pragmatic contrast — Route B only ===')
print(f"Contrast rows: {len(select_pragmatic_contrast_rows(manifest))}")
fb1 = aim1_rb[aim1_rb['file_backed_rna']]
print(f"File-backed (direct or Route B): {len(fb1)}")
if len(fb1):
    print(pd.crosstab(fb1['cohort'], fb1['tcab1_overall_status']).to_string())
    print(pd.crosstab(fb1['recovery_route'], fb1['tcab1_overall_status']).to_string())
print()

print('=== Aim 2 ALT classifier — Route B only ===')
print(f"Contrast rows: {len(select_aim2_contrast_rows(manifest))}")
fb2 = aim2_rb[aim2_rb['file_backed_rna']]
print(f"File-backed (direct or Route B): {len(fb2)}")
if len(fb2):
    print(pd.crosstab(fb2['cohort'], fb2['alt_status']).to_string())
    print(pd.crosstab(fb2['recovery_route'], fb2['alt_status']).to_string())
print()

assess_aim1 = assess_confounding_generic(
    aim1_rb, 'tcab1_overall_status', ('disrupted', 'intact'),
)
assess_aim2 = assess_confounding_generic(
    aim2_rb, 'alt_status', (ALT_POSITIVE_LABEL, ALT_NEGATIVE_LABEL),
)
verdict_aim1 = decide_route_b_aim1(aim1_rb, assess_aim1)
verdict_aim2 = decide_route_b_aim2(aim2_rb, assess_aim2)

print(f"Aim 1 Route-B verdict: {verdict_aim1['category']}")
print(verdict_aim1['justification'])
print()
print(f"Aim 2 Route-B verdict: {verdict_aim2['category']}")
print(verdict_aim2['justification'])

=== Aim 1 pragmatic contrast — Route B only ===
Contrast rows: 40
File-backed (direct or Route B): 2
tcab1_overall_status  disrupted
cohort                         
DepMap_26Q1                   2
tcab1_overall_status  disrupted
recovery_route                 
direct+route_b                2

=== Aim 2 ALT classifier — Route B only ===
Contrast rows: 59
File-backed (direct or Route B): 7
alt_status         alt_gold_positive  alt_negative_high_conf
cohort                                                      
Lawlor_2026_GCCRI                  0                       1
Taglialatela_2026                  4                       2
alt_status      alt_gold_positive  alt_negative_high_conf
recovery_route                                           
route_b                         4                       3

Aim 1 Route-B verdict: not_estimable
File-backed RNA is available for 2 disrupted sample(s) (from cohort(s) DepMap_26Q1) and 0 intact sample(s) (from cohort(s) (none)). TCAB1 status is perfe

## 6 — Write `data/processed/notebook_08_report.md`

In [7]:
REPORT_PATH = ROOT / 'data' / 'processed' / 'notebook_08_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

VERDICT_LABEL = {
    'within_cohort_viable': 'WITHIN-COHORT VIABLE',
    'pooled_exploratory_only': 'POOLED-EXPLORATORY ONLY',
    'not_estimable': 'NOT ESTIMABLE AS DEFINED',
}

def _md_table(df, index_label):
    if df.empty:
        return '(no rows)\n'
    cols = list(df.columns)
    header = '| ' + index_label + ' | ' + ' | '.join(str(c) for c in cols) + ' |\n'
    sep = '|' + '|'.join(['---'] * (len(cols) + 1)) + '|\n'
    body = ''.join(
        '| ' + str(idx) + ' | ' + ' | '.join(str(v) for v in row) + ' |\n'
        for idx, row in df.iterrows()
    )
    return header + sep + body

lines: list[str] = []
lines.append('# Notebook 08 — Recovery Route Inventory\n')
lines.append(f'_Generated: {RUN_TIMESTAMP}_\n')
lines.append('')
lines.append(
    'Read-only diagnostic testing three RNA-recovery routes not evaluated in '
    'notebooks 06/07 (exact `sample_id` matching only).\n'
)
lines.append('')

lines.append('## Route B — DepMap cell-line-name join\n')
lines.append(f'- Candidates: **{len(route_b_candidates)}** '
             f'(Taglialatela + ALTitude DepMap panel overlap).\n')
lines.append(f'- Mapped to ModelID: **{rb_counts["n_mapped_total"]}**\n')
lines.append(f'- In DepMap RNA matrix: **{rb_counts["n_in_depmap_rna"]}**\n')
lines.append(f'- ALT-positive + RNA-backed: **{rb_counts["n_alt_positive_rna"]}**\n')
lines.append(f'- ALT-negative + RNA-backed: **{rb_counts["n_alt_negative_rna"]}**\n')
lines.append('\n### Route B mapping table\n')
lines.append('| cell_line_name | mapped_model_id | in_depmap_rna | alt_status | source | cohort |\n')
lines.append('|---|---|---|---|---|---|\n')
for _, r in route_b.iterrows():
    lines.append(
        f"| {r['cell_line_name']} | {r['mapped_model_id']} | {r['in_depmap_rna']} | "
        f"{r['alt_status']} | {r['source']} | {r['cohort']} |"
    )
lines.append('')

lines.append('## Route A — de Nonneville (E-MTAB-11743)\n')
lines.append(f"- Manifest samples with ALT labels: **{route_a['n_total']}** "
             f"(positive={route_a['n_alt_positive']}, negative={route_a['n_alt_negative']}).\n")
lines.append(f"- Expression deposited at ArrayExpress **{ARRAYEXPRESS_DE_NONNEVILLE}** "
             '(not downloaded in this notebook).\n')
lines.append('- **Human verification required:** confirm assay type is RNA-seq, not methylation array.\n')
lines.append('\n```\n' + ROUTE_A_TODO.strip() + '\n```\n')

lines.append('## Route C — Lawlor internal PDX RNA\n')
lines.append(f"- Lawlor PDX rows in manifest: **{route_c['n_total_lawlor']}**\n")
lines.append(f"- With ALT and/or architecture labels: **{route_c['n_with_alt_or_architecture']}** "
             f"(ALT-positive={route_c['n_alt_positive']}, "
             f"ALT-negative={route_c['n_alt_negative']}, "
             f"architecture-labeled={len(route_c['sample_ids_architecture_labeled'])}).\n")
lines.append('- RNA is **not public**; requires internal lab access.\n')
lines.append('\n```\n' + ROUTE_C_TODO.strip() + '\n```\n')

lines.append('## Combined — Route-B-only best case\n')
lines.append('Assumes Route B succeeds; Routes A and C remain pending.\n')
lines.append('\n### Aim 1 (TCAB1 pragmatic contrast)\n')
lines.append(f"- File-backed RNA samples: **{assess_aim1['n_total']}** "
             f"(disrupted={assess_aim1['n_arm0']}, intact={assess_aim1['n_arm1']}).\n")
if not assess_aim1['cohort_x_arm'].empty:
    lines.append(_md_table(assess_aim1['cohort_x_arm'], 'cohort'))
lines.append(f"- Verdict: **{VERDICT_LABEL[verdict_aim1['category']]}**\n")
lines.append(f"> {verdict_aim1['justification']}\n")
lines.append('\n### Aim 2 (ALT classifier contrast)\n')
lines.append(f"- File-backed RNA samples: **{assess_aim2['n_total']}** "
             f"(positive={assess_aim2['n_arm0']}, negative={assess_aim2['n_arm1']}).\n")
if not assess_aim2['cohort_x_arm'].empty:
    lines.append(_md_table(assess_aim2['cohort_x_arm'], 'cohort'))
lines.append(f"- Cohorts with both arms (same data_type): `{assess_aim2['cohorts_with_both_arms_same_dtype']}`\n")
lines.append(f"- Verdict: **{VERDICT_LABEL[verdict_aim2['category']]}**\n")
lines.append(f"> {verdict_aim2['justification']}\n")
lines.append('')

lines.append('## RECOVERY VERDICT\n')
lines.append(
    '**Route B (achievable now, zero external dependency):** '
    f"cell-line join recovers **{rb_counts['n_in_depmap_rna']}** DepMap RNA profiles "
    f"({rb_counts['n_alt_positive_rna']} ALT-positive, {rb_counts['n_alt_negative_rna']} ALT-negative "
    'among candidates with gold ALT calls). '
    f"Within the Aim 2 primary contrast, Route B alone yields **{assess_aim2['n_total']}** "
    f"file-backed samples ({assess_aim2['n_arm0']} positive, {assess_aim2['n_arm1']} negative) "
    f"— verdict **{VERDICT_LABEL[verdict_aim2['category']]}** "
    f"(within-cohort minimum = {WITHIN_COHORT_PER_ARM_MIN} per arm). "
    f"Aim 1 remains **{VERDICT_LABEL[verdict_aim1['category']]}** "
    f"({assess_aim1['n_arm0']} disrupted, {assess_aim1['n_arm1']} intact file-backed).\n"
)
lines.append('')
lines.append(
    '**Routes A and C (pending):** de Nonneville (`E-MTAB-11743`, 22 labeled tumors) and '
    f"Lawlor ({route_c['n_with_alt_or_architecture']} labeled PDX models) would add the largest "
    'labeled cohorts if expression is obtained and sample IDs mapped. Neither was counted '
    'in the Route-B-only verdict above.\n'
)
lines.append('')
lines.append('**Recommendation:**\n')
lines.append(
    '1. Proceed with Route B integration for Taglialatela cell lines (6/7 map to DepMap RNA; '
    'SKLU1 maps to ACH-000309 but is absent from the expression matrix).\n'
)
lines.append('2. Inspect E-MTAB-11743 assay metadata before downloading (Route A).\n')
lines.append('3. Request Lawlor PDX RNA from the lab (Route C) — highest leverage for both aims.\n')
lines.append('')

REPORT_PATH.write_text('\n'.join(lines))
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')

Wrote data/processed/notebook_08_report.md  (8889 bytes)


## 7 — Idempotence check

In [8]:
import hashlib

first_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
REPORT_PATH.write_text('\n'.join(lines))
second_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
assert first_hash == second_hash, 'Notebook 08 is not idempotent'
print('Idempotent: re-writing produces identical bytes.')

Idempotent: re-writing produces identical bytes.
